In [0]:
storage_account_name = "f1datalakestorageme"
storage_key = dbutils.secrets.get(scope="storage-secrets", key="f1-storage-account-key")

# Inject authentication directly into the Spark session
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_key)


In [0]:

# COMMAND ----------

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, explode, current_timestamp, concat, lit
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, ArrayType

# COMMAND ----------

# 1. Outer Schema Enforcement
races_schema = StructType(fields=[
    StructField("MRData", StructType([
        StructField("RaceTable", StructType([
            StructField("Races", StringType(), True)
        ]), True)
    ]), True)
])

raw_races_df = spark.read \
    .schema(races_schema) \
    .json("dbfs:/mnt/f1-raw/races/*")

# COMMAND ----------

# 2. Inner Array Schema Definition & Transformation
race_element_schema = StructType([
    StructField("season", StringType(), True),
    StructField("round", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", StringType(), True),
    StructField("Circuit", StructType([StructField("circuitId", StringType(), True)]), True)
])

parsed_df = raw_races_df.withColumn(
    "race_array", 
    from_json(col("MRData.RaceTable.Races"), ArrayType(race_element_schema))
)
exploded_df = parsed_df.select(explode(col("race_array")).alias("race"))

# Generate structured fields and build a reliable business key (race_id)
silver_races_df = exploded_df.select(
    concat(col("race.season"), lit("_"), col("race.round")).alias("race_id"),
    col("race.season").cast("int").alias("race_year"),
    col("race.round").cast("int").alias("round"),
    col("race.raceName").alias("name"),
    col("race.date").cast("date").alias("race_date"),
    col("race.Circuit.circuitId").alias("circuit_id"),
    current_timestamp().alias("ingestion_date")
).dropDuplicates(["race_id"])

display(silver_races_df)

# COMMAND ----------

# 3. Idempotent Upsert into Silver Layer
target_path = "dbfs:/mnt/f1-transformed/races"
spark.sql("CREATE DATABASE IF NOT EXISTS hive_metastore.f1_transformed")
# COMMAND ----------

# 3. Unity Catalog Compliant Table Write & Upsert

# Ensure the schema/database container exists inside Hive Metastore
spark.sql("CREATE DATABASE IF NOT EXISTS hive_metastore.f1_transformed")

# Check if target catalog table exists
if not spark.catalog.tableExists("hive_metastore.f1_transformed.races"):
    # First-time initialization run:
    # Save directly as a Hive Metastore Managed Table
    silver_races_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("hive_metastore.f1_transformed.races")
    print("Races table successfully initialized as a Hive Metastore Managed Table.")
else:
    # Production Merge Routine using catalog table reference directly
    tgt_table = DeltaTable.forName(spark, "hive_metastore.f1_transformed.races")
    
    tgt_table.alias("tgt") \
        .merge(
            source = silver_races_df.alias("src"), 
            condition = "tgt.race_id = src.race_id"
        ) \
        .whenMatchedUpdate(set = {
            "race_year": "src.race_year", 
            "round": "src.round", 
            "name": "src.name", 
            "race_date": "src.race_date", 
            "circuit_id": "src.circuit_id", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .whenNotMatchedInsert(values = {
            "race_id": "src.race_id", 
            "race_year": "src.race_year", 
            "round": "src.round", 
            "name": "src.name", 
            "race_date": "src.race_date", 
            "circuit_id": "src.circuit_id", 
            "ingestion_date": "src.ingestion_date"
        }) \
        .execute()
    print("Races incremental update completed successfully via Hive Metastore Delta Merge.")

In [0]:
# Force clean up the broken pipe
if any(mount.mountPoint == "/mnt/f1-raw" for mount in dbutils.fs.mounts()):
    dbutils.fs.unmount("/mnt/f1-raw")
